# LangSmith 新版本交互式学习笔记

这份 Notebook 用来学习新版 LangSmith 的常用能力：Tracing、`@traceable`、`tracing_context`、LangChain 自动追踪、metadata/tags、项目名、以及评测 Evaluation 的基本写法。

使用建议：

- 本项目使用 `uv` 管理依赖；不要在 Notebook 里直接写 `pip install ...`。
- 如果缺依赖，推荐在项目根目录执行 `uv add langsmith` 或 `uv sync`。
- 真正上传 trace 需要配置 `LANGSMITH_API_KEY`。
- 没配置 LangSmith Key 时，带外部上传的单元会自动跳过，不会硬报错。
- DeepSeek 调用读取项目根目录 `.env` 里的 `DEEPSEEK_API_KEY` / `OPENAI_API_KEY` 和 `DEEPSEEK_MODEL` / `OPENAI_MODEL`。

In [ ]:
import os
import sys
from pathlib import Path


def find_project_root(start: str | Path | None = None) -> Path:
    """从当前目录向上查找项目根目录，优先识别 .env / pyproject.toml。"""
    current = Path(start or Path.cwd()).resolve()
    candidates = [current, *current.parents]

    for path in candidates:
        if (path / ".env").exists() and (path / "pyproject.toml").exists():
            return path

    for path in candidates:
        if (path / ".env").exists():
            return path

    return current


def load_project_env() -> Path | None:
    """加载项目根目录 .env；Notebook 位于 docs/ 时也能找到根目录配置。"""
    project_root = find_project_root()
    env_path = project_root / ".env"

    if not env_path.exists():
        print(f"未找到 .env，当前查找起点：{Path.cwd()}")
        return None

    try:
        from dotenv import load_dotenv
    except ImportError:
        print("未安装 python-dotenv，跳过 .env 加载。")
        return env_path

    load_dotenv(env_path, override=False)
    print(f"已加载 .env：{env_path}")
    return env_path

load_project_env()

print("Python:", sys.version)
print("当前解释器:", sys.executable)
print("当前目录:", Path.cwd())
print("项目根目录:", find_project_root())
print("LANGSMITH_API_KEY 是否存在:", bool(os.getenv("LANGSMITH_API_KEY")))
print("LANGSMITH_PROJECT:", os.getenv("LANGSMITH_PROJECT") or "learnone-langsmith-notebook")
print("LANGSMITH_TRACING:", os.getenv("LANGSMITH_TRACING"))

## 1. LangSmith 是什么

LangSmith 是 LangChain 官方的可观测性、调试和评测平台。它常用于：

- 查看一次 LLM 应用调用经历了哪些步骤。
- 对比 Prompt、模型、参数变化对输出的影响。
- 追踪 Agent 的工具调用、RAG 检索、模型响应。
- 给线上或实验应用添加 tags、metadata、project，便于后续筛选。
- 做数据集评测和实验对比。

新版核心配置通常是：

```text
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=你的 LangSmith Key
LANGSMITH_PROJECT=项目名，可选
```

如果不想全局打开 tracing，也可以在代码里用 `tracing_context(enabled=True)` 临时开启。

In [ ]:
# 本项目推荐用 uv 管理依赖。这个单元不会自动安装，只打印建议命令。
print("如果缺少 langsmith，请在项目根目录执行：")
print("uv add langsmith")
print("uv sync")

# 如果 langsmith 已经由 langchain 间接安装，也可以直接使用。
import importlib.util
print("langsmith 可导入:", importlib.util.find_spec("langsmith") is not None)

## 2. 环境变量与开关

LangSmith tracing 是否上传，主要看这些环境变量：

| 变量 | 作用 |
| --- | --- |
| `LANGSMITH_API_KEY` | LangSmith 平台 API Key |
| `LANGSMITH_TRACING` | 是否启用 tracing，通常设为 `true` |
| `LANGSMITH_PROJECT` | trace 归属项目名 |
| `LANGSMITH_ENDPOINT` | LangSmith API 地址，通常不用改 |
| `LANGSMITH_WORKSPACE_ID` | 多 workspace 时指定工作区，可选 |

下面单元会设置默认项目名，但不会伪造 API Key。

In [ ]:
import os
import sys
from pathlib import Path


def find_project_root(start: str | Path | None = None) -> Path:
    """从当前目录向上查找项目根目录，优先识别 .env / pyproject.toml。"""
    current = Path(start or Path.cwd()).resolve()
    candidates = [current, *current.parents]

    for path in candidates:
        if (path / ".env").exists() and (path / "pyproject.toml").exists():
            return path

    for path in candidates:
        if (path / ".env").exists():
            return path

    return current


def load_project_env() -> Path | None:
    """加载项目根目录 .env；Notebook 位于 docs/ 时也能找到根目录配置。"""
    project_root = find_project_root()
    env_path = project_root / ".env"

    if not env_path.exists():
        print(f"未找到 .env，当前查找起点：{Path.cwd()}")
        return None

    try:
        from dotenv import load_dotenv
    except ImportError:
        print("未安装 python-dotenv，跳过 .env 加载。")
        return env_path

    load_dotenv(env_path, override=False)
    print(f"已加载 .env：{env_path}")
    return env_path

load_project_env()

# 如果 .env 没写项目名，这里给 Notebook 设置一个默认项目名。
os.environ.setdefault("LANGSMITH_PROJECT", "learnone-langsmith-notebook")

# 只有存在 API Key 时才默认打开 tracing，避免没有 Key 时不断报上传失败。
if os.getenv("LANGSMITH_API_KEY"):
    os.environ.setdefault("LANGSMITH_TRACING", "true")

print("LANGSMITH_API_KEY 是否存在:", bool(os.getenv("LANGSMITH_API_KEY")))
print("LANGSMITH_TRACING:", os.getenv("LANGSMITH_TRACING"))
print("LANGSMITH_PROJECT:", os.getenv("LANGSMITH_PROJECT"))

## 3. `@traceable`：最小可追踪函数

`@traceable` 是 Python SDK 里最常用的自定义追踪方式。它会把函数调用记录成 LangSmith 里的一个 run。

这个示例在没有 `LANGSMITH_API_KEY` 时仍然能运行，只是不会上传到 LangSmith。

In [ ]:
from langsmith import traceable


# @traceable 会把函数调用记录成 LangSmith run。
# name 是 run 名称，run_type 用来标记这类调用属于 chain / llm / tool / retriever 等。
@traceable(name="normalize_question", run_type="chain", tags=["notebook", "basic"])
def normalize_question(question: str) -> str:
    """清理用户问题。"""
    # strip 去掉两端空白，replace 把中文问号统一成英文问号。
    return question.strip().replace("？", "?")


print(normalize_question("  什么是 LangSmith？  "))


## 4. 嵌套 trace：chain / tool / retriever

LangSmith 的可观测性价值在于能看到嵌套调用：一个 pipeline 里面调用了检索、工具、模型等多个步骤。

下面用纯 Python 离线模拟一个 RAG pipeline。

In [ ]:
from langsmith import traceable

DOCS = [
    "LangSmith 用于追踪、调试、评测 LLM 应用。",
    "LangChain 用于构建 Prompt、Tool、Agent、RAG 等组件。",
    "RAG 会先检索资料，再让模型基于资料回答。",
]


@traceable(name="retrieve_docs", run_type="retriever", tags=["notebook", "rag"])
def retrieve_docs(query: str) -> list[str]:
    # 用关键词重叠模拟检索分数，真实项目会换成向量库或搜索服务。
    keywords = set(query.lower().split())
    scored = []
    for doc in DOCS:
        score = sum(1 for word in keywords if word in doc.lower())
        scored.append((score, doc))
    # 返回分数最高的前 2 条文档。
    return [doc for score, doc in sorted(scored, reverse=True)[:2]]


@traceable(name="format_context", run_type="chain")
def format_context(docs: list[str]) -> str:
    # 把检索结果拼成 Prompt 上下文。
    return "
".join(f"- {doc}" for doc in docs)


@traceable(name="offline_answer", run_type="llm")
def offline_answer(question: str, context: str) -> str:
    # 离线模拟 LLM 回答，避免没有 API Key 时 Notebook 报错。
    return f"基于资料回答：{question}

参考资料：
{context}"


@traceable(name="rag_pipeline", run_type="chain", tags=["notebook", "pipeline"])
def rag_pipeline(question: str) -> str:
    # 嵌套调用会在 LangSmith 中形成父子 run 树。
    docs = retrieve_docs(question)
    context = format_context(docs)
    return offline_answer(question, context)


print(rag_pipeline("LangSmith RAG 追踪"))


## 5. `tracing_context`：临时控制是否追踪

新版 LangSmith 支持用 `tracing_context(enabled=True/False)` 在局部代码块里控制追踪，而不是完全依赖全局环境变量。

适合场景：

- 本地调试时临时打开。
- 单元测试时关闭。
- 只追踪关键链路，避免噪音太多。

In [ ]:
from langsmith import tracing_context

# tracing_context 可以临时控制当前代码块是否开启追踪。
# 即使全局 LANGSMITH_TRACING 没打开，也可以在这个上下文里临时启用。
# 如果没有 LANGSMITH_API_KEY，调用仍可本地运行，但不会成功上传到平台。
with tracing_context(enabled=bool(os.getenv("LANGSMITH_API_KEY"))):
    result = rag_pipeline("LangSmith 如何追踪嵌套调用？")
    print(result[:120], "...")


## 6. metadata 与 tags

Tags 用于筛选，metadata 用于保存结构化上下文，比如用户 ID、环境、版本、实验参数等。

注意不要把 API Key、手机号、身份证号、完整隐私文本放进 metadata。

In [ ]:
from langsmith import traceable


@traceable(
    name="classify_intent",
    run_type="chain",
    # tags 适合用来筛选 run，例如 notebook、intent、deepseek。
    tags=["notebook", "intent"],
    # metadata 适合记录版本、环境、课程、用户类型等结构化信息。
    metadata={"env": "local", "lesson": "langsmith"},
)
def classify_intent(text: str) -> dict:
    # 这里用简单规则模拟意图分类。
    if "天气" in text:
        intent = "weather"
    elif "LangSmith" in text:
        intent = "langsmith"
    else:
        intent = "other"
    return {"text": text, "intent": intent}


print(classify_intent("LangSmith 怎么用？"))


## 7. 当前 run：在函数内部写入更多信息

在 traceable 函数内部，可以通过 `get_current_run_tree()` 获取当前 run，并追加 metadata 或 tags。这样可以把运行时才知道的信息写进 trace。

In [ ]:
from langsmith.run_helpers import get_current_run_tree
from langsmith import traceable


@traceable(name="pipeline_with_runtime_metadata", run_type="chain")
def pipeline_with_runtime_metadata(question: str) -> str:
    # get_current_run_tree 可以在被追踪函数内部拿到当前 run。
    run = get_current_run_tree()
    if run:
        # 运行时动态追加 metadata 和 tags。
        run.add_metadata({"question_length": len(question)})
        run.add_tags(["runtime-metadata"])
    return question.upper()


print(pipeline_with_runtime_metadata("hello langsmith"))


## 8. LangChain 自动追踪 DeepSeek 调用

LangChain 和 LangSmith 集成很紧密：当 `LANGSMITH_TRACING=true` 且配置了 `LANGSMITH_API_KEY` 时，LangChain 的模型、Prompt、链、Agent 调用会自动出现在 LangSmith 中。

下面示例使用你项目里的 DeepSeek 配置。没有 DeepSeek Key 或没有 LangSmith Key 时会自动跳过真实上传。

In [ ]:
import os
import sys
from pathlib import Path


def find_project_root(start: str | Path | None = None) -> Path:
    """从当前目录向上查找项目根目录，优先识别 .env / pyproject.toml。"""
    current = Path(start or Path.cwd()).resolve()
    candidates = [current, *current.parents]

    for path in candidates:
        if (path / ".env").exists() and (path / "pyproject.toml").exists():
            return path

    for path in candidates:
        if (path / ".env").exists():
            return path

    return current


def load_project_env() -> Path | None:
    """加载项目根目录 .env；Notebook 位于 docs/ 时也能找到根目录配置。"""
    project_root = find_project_root()
    env_path = project_root / ".env"

    if not env_path.exists():
        print(f"未找到 .env，当前查找起点：{Path.cwd()}")
        return None

    try:
        from dotenv import load_dotenv
    except ImportError:
        print("未安装 python-dotenv，跳过 .env 加载。")
        return env_path

    load_dotenv(env_path, override=False)
    print(f"已加载 .env：{env_path}")
    return env_path

load_project_env()
os.environ.setdefault("LANGSMITH_PROJECT", "learnone-langsmith-notebook")
if os.getenv("LANGSMITH_API_KEY"):
    os.environ.setdefault("LANGSMITH_TRACING", "true")


def get_deepseek_model(temperature: float = 0):
    api_key = os.getenv("DEEPSEEK_API_KEY") or os.getenv("OPENAI_API_KEY")
    model_name = os.getenv("DEEPSEEK_MODEL") or os.getenv("OPENAI_MODEL") or "deepseek-chat"

    if not api_key:
        print("未配置 DEEPSEEK_API_KEY 或 OPENAI_API_KEY，跳过真实模型调用。")
        return None

    try:
        from langchain_deepseek import ChatDeepSeek
    except ImportError:
        print("未安装 langchain-deepseek，跳过真实模型调用。")
        return None

    return ChatDeepSeek(model=model_name, api_key=api_key, temperature=temperature)


model = get_deepseek_model(temperature=0)
print("DeepSeek model:", type(model).__name__ if model else None)
print("LangSmith tracing:", os.getenv("LANGSMITH_TRACING"))
print("LangSmith key:", bool(os.getenv("LANGSMITH_API_KEY")))

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

if model and os.getenv("LANGSMITH_API_KEY"):
    # Prompt -> DeepSeek model -> StrOutputParser 是最常见的 LangChain 链。
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "你是一个简洁的 LangSmith 教学助手。"),
            ("human", "请用三句话解释：{topic}"),
        ]
    )
    chain = prompt | model | StrOutputParser()

    # 当 LANGSMITH_TRACING=true 且有 API Key 时，这次调用会被自动追踪。
    answer = chain.invoke({"topic": "LangSmith tracing"})
    print(answer)
else:
    print("缺少 DeepSeek 模型或 LANGSMITH_API_KEY，跳过真实 LangChain trace 上传。")


## 9. 给 LangChain 调用动态添加 project / tags / metadata

除了环境变量，也可以在 `invoke(..., config={...})` 中传入 tags、metadata、run_name 等信息。这样同一条链可以按不同实验、用户、功能做筛选。

In [ ]:
if model and os.getenv("LANGSMITH_API_KEY"):
    prompt = ChatPromptTemplate.from_template("请解释：{topic}")
    chain = prompt | model | StrOutputParser()

    answer = chain.invoke(
        {"topic": "LangSmith metadata"},
        config={
            # run_name 会显示在 LangSmith trace 中。
            "run_name": "deepseek_langsmith_metadata_demo",
            # tags 方便按 notebook / deepseek / metadata 筛选。
            "tags": ["notebook", "deepseek", "metadata"],
            # metadata 记录结构化上下文，便于后续分析。
            "metadata": {
                "course": "learnone",
                "provider": "deepseek",
                "model": os.getenv("DEEPSEEK_MODEL") or os.getenv("OPENAI_MODEL"),
            },
        },
    )
    print(answer)
else:
    print("缺少 DeepSeek 模型或 LANGSMITH_API_KEY，跳过动态 metadata 示例。")


## 10. `Client`：连接 LangSmith 平台

`Client` 用于和 LangSmith 平台交互，比如检查项目、创建数据集、上传 examples、查看 runs 等。

下面只在配置了 `LANGSMITH_API_KEY` 时调用平台。

In [ ]:
from langsmith import Client

if os.getenv("LANGSMITH_API_KEY"):
    # Client 用来访问 LangSmith 平台能力，例如 datasets、examples、runs、projects。
    client = Client()
    print("Client 创建成功")
    print("项目名:", os.getenv("LANGSMITH_PROJECT"))
else:
    client = None
    print("未配置 LANGSMITH_API_KEY，跳过 Client 平台调用。")


## 11. Evaluation 基础概念

LangSmith Evaluation 的基本对象：

- Dataset：一组测试样例。
- Example：输入、期望输出或参考答案。
- Target function：被评测的函数或链。
- Evaluator：评分函数，比如是否包含关键词、是否和参考答案一致。
- Experiment：一次评测运行。

为了避免误创建远程数据集，下面默认只演示本地 evaluator 写法；真正调用 `evaluate()` 的单元加了 Key 检查。

In [ ]:
def target_app(inputs: dict) -> dict:
    # target_app 表示被评测的应用。输入通常来自 dataset example.inputs。
    question = inputs["question"]
    if "LangSmith" in question:
        answer = "LangSmith 用于追踪、调试和评测 LLM 应用。"
    else:
        answer = "我不知道。"
    return {"answer": answer}


def contains_expected(outputs: dict, reference_outputs: dict) -> bool:
    # 一个最简单的 evaluator：判断答案是否包含期望关键词。
    expected = reference_outputs["must_contain"]
    return expected in outputs["answer"]


example = {
    "inputs": {"question": "LangSmith 是什么？"},
    "reference_outputs": {"must_contain": "追踪"},
}

outputs = target_app(example["inputs"])
print(outputs)
print("通过:", contains_expected(outputs, example["reference_outputs"]))


In [ ]:
# 可选：真实 LangSmith evaluate，会在平台产生实验记录。
# 如果你只是学习，不想创建远程实验，可以跳过这个单元。
if os.getenv("LANGSMITH_API_KEY"):
    try:
        from langsmith import evaluate

        # dataset 这里用内存列表演示；真实项目也可以使用 LangSmith 平台 Dataset。
        dataset = [
            {
                "inputs": {"question": "LangSmith 是什么？"},
                "outputs": {"must_contain": "追踪"},
            },
            {
                "inputs": {"question": "LangChain 是什么？"},
                "outputs": {"must_contain": "我不知道"},
            },
        ]

        def evaluator(run, example) -> dict:
            # run.outputs 是 target_app 的实际输出。
            answer = run.outputs["answer"]
            # example.outputs 是期望结果。
            expected = example.outputs["must_contain"]
            return {"key": "contains_expected", "score": expected in answer}

        results = evaluate(
            target_app,
            data=dataset,
            evaluators=[evaluator],
            experiment_prefix="learnone-langsmith-notebook",
        )
        print(results)
    except Exception as exc:
        print("evaluate 示例未执行成功，请检查当前 langsmith 版本和 API Key。")
        print(type(exc).__name__, exc)
else:
    print("未配置 LANGSMITH_API_KEY，跳过真实 evaluate 示例。")


## 12. 常见排查

如果 LangSmith 没有 trace，优先检查：

1. 当前 Notebook kernel 是不是项目 `.venv`。
2. `.env` 是否在项目根目录，是否能被加载。
3. 是否配置了 `LANGSMITH_API_KEY`。
4. 是否设置了 `LANGSMITH_TRACING=true`。
5. `LANGSMITH_PROJECT` 是否写到了你正在查看的项目。
6. 程序是否太快退出，导致后台 trace 还没提交完成。
7. 是否不小心把 tracing 关在了 `tracing_context(enabled=False)` 里。

Notebook 里不要打印 API Key，只打印是否存在即可。

In [ ]:
import os
import sys
from pathlib import Path


def find_project_root(start: str | Path | None = None) -> Path:
    """从当前目录向上查找项目根目录，优先识别 .env / pyproject.toml。"""
    current = Path(start or Path.cwd()).resolve()
    candidates = [current, *current.parents]

    for path in candidates:
        if (path / ".env").exists() and (path / "pyproject.toml").exists():
            return path

    for path in candidates:
        if (path / ".env").exists():
            return path

    return current


def load_project_env() -> Path | None:
    """加载项目根目录 .env；Notebook 位于 docs/ 时也能找到根目录配置。"""
    project_root = find_project_root()
    env_path = project_root / ".env"

    if not env_path.exists():
        print(f"未找到 .env，当前查找起点：{Path.cwd()}")
        return None

    try:
        from dotenv import load_dotenv
    except ImportError:
        print("未安装 python-dotenv，跳过 .env 加载。")
        return env_path

    load_dotenv(env_path, override=False)
    print(f"已加载 .env：{env_path}")
    return env_path

load_project_env()

checks = {
    "当前解释器": sys.executable,
    "LANGSMITH_API_KEY": bool(os.getenv("LANGSMITH_API_KEY")),
    "LANGSMITH_TRACING": os.getenv("LANGSMITH_TRACING"),
    "LANGSMITH_PROJECT": os.getenv("LANGSMITH_PROJECT"),
    "DEEPSEEK_API_KEY": bool(os.getenv("DEEPSEEK_API_KEY")),
    "OPENAI_API_KEY": bool(os.getenv("OPENAI_API_KEY")),
    "DEEPSEEK_MODEL": os.getenv("DEEPSEEK_MODEL") or os.getenv("OPENAI_MODEL"),
}

for key, value in checks.items():
    print(f"{key}: {value}")

In [ ]:
# Notebook / 短脚本里可以尝试等待 LangSmith 后台提交完成。
# 不同版本 SDK 的导入路径可能不同，所以这里做兼容保护。
try:
    from langsmith.run_helpers import wait_for_all_tracers
except ImportError:
    wait_for_all_tracers = None

if wait_for_all_tracers:
    wait_for_all_tracers()
    print("已等待 LangSmith tracer 提交完成。")
else:
    print("当前 SDK 没有暴露 wait_for_all_tracers；Notebook 学习场景可直接跳过。")


## 13. 官方参考

- LangSmith + LangChain tracing: https://docs.langchain.com/langsmith/trace-with-langchain
- Trace without env vars: https://docs.langchain.com/langsmith/trace-without-env-vars
- Distributed tracing: https://docs.langchain.com/langsmith/distributed-tracing
- `traceable` API: https://reference.langchain.com/python/langsmith/run_helpers/traceable
- `tracing_context` API: https://reference.langchain.com/python/langsmith/run_helpers/tracing_context
- LangSmith Python SDK reference: https://reference.langchain.com/python/langsmith